# Five reference queries against `buildings.parquet`

Run after `plateau build --city 13113 --out ./out`.

These are the queries downstream apps (prettyplateau, Creative MCP, Risk Lens) issue most often. If you can run them all, your parquet is healthy.

In [ ]:
import duckdb
import geopandas as gpd

PARQUET = 'out/buildings.parquet'

## 1. Filter by attributes

All wooden buildings older than 1981 (旧耐震) in 渋谷区. This is the basis for `prettyplateau`'s *Surviving Buildings* poster.

In [ ]:
duckdb.sql(f"""
  SELECT building_uid, year_built, structure
  FROM '{PARQUET}'
  WHERE city_code = '13113' AND structure = 'wood' AND year_built < 1981
  LIMIT 10
""").df()

## 2. colorBy: decade histogram

Year-built distribution. `prettyplateau`'s *Building Age Rainbow* is this same query, drawn instead of tabulated.

In [ ]:
duckdb.sql(f"""
  SELECT FLOOR(year_built / 10) * 10 AS decade, COUNT(*) AS n
  FROM '{PARQUET}'
  WHERE year_built IS NOT NULL
  GROUP BY 1 ORDER BY 1
""").df()

## 3. Hazard intersection (the honesty test)

Two related questions. Note how they differ — and why the difference matters.

In [ ]:
duckdb.sql(f"""
  SELECT
    SUM(CASE WHEN river_flood_covered AND river_flood_depth_max > 1.0 THEN 1 ELSE 0 END) AS at_risk,
    SUM(CASE WHEN river_flood_covered AND COALESCE(river_flood_depth_max, 0) = 0 THEN 1 ELSE 0 END) AS surveyed_safe,
    SUM(CASE WHEN NOT river_flood_covered THEN 1 ELSE 0 END) AS unknown
  FROM '{PARQUET}'
""").df()

The third column (`unknown`) is what makes this dataset trustworthy. A naïve pipeline would silently bucket those buildings as *safe* — we keep them explicitly separate, and surface them as grey in any UI.

## 4. Centroid table (for OSM fusion at runtime)

We never redistribute OSM-fused parquet, but downstream apps can JOIN against OSM on the user's machine. Centroids are the join key.

In [ ]:
duckdb.sql(f"""
  SELECT building_uid, centroid_lon, centroid_lat
  FROM '{PARQUET}' LIMIT 5
""").df()

## 5. bbox query (Risk Lens export path)

User draws a bounding box in the browser → server cuts a slice. For full precision use the FlatGeobuf per-ward files; for SQL convenience the parquet `.cx` selector also works.

In [ ]:
gdf = gpd.read_parquet(PARQUET)
sub = gdf.cx[139.70:139.71, 35.65:35.66]
print(f'{len(sub)} buildings in bbox')
sub.head()